# Background Image Extraction Notebook

## Overview

This notebook extracts background (without dead cell) image patches from a dataset of fluorescent microscopy images. The workflow uses a sliding-window approach with overlapping crops to systematically sample regions that contain no cells, filtering by image darkness (mean pixel intensity).

### Workflow
1. Load a dataset and select preprocessing parameters
2. Iterate through source images (Green and Phase channels)
3. Apply sliding-window cropping with edge coverage
4. Process each crop with the image pipeline
5. Save background crops (no cell detections, sufficiently dark)

### Output
- **Location**: `Background/` directory under the dataset path
- **Format**: PNG images, 128×128 pixels
- **Naming**: `{original_name}_{window_x}_{window_y}.png`

### Requirements
- OpenCV (`cv2`)
- NumPy
- `CellProcessor` module (custom image processing utilities)

In [21]:
"""Import required libraries for image processing and dataset management."""
import cv2
import os
import numpy as np
from CellProcessor import (
    read_image, 
    process_image, 
    get_bboxes, 
    use_dataset, 
    list_dataset,
    use_variables, 
    list_variables
)

## Configuration
Select the dataset and preprocessing parameters to use.

In [22]:
"""List available datasets and select one to use."""
print("Available datasets:")
list_dataset()

Available datasets:


,ID,Cell_type,Death_type,Image_path,Description
0,1,MEF,Necroptosis,Data,Test


In [33]:
"""List available preprocessing variable sets."""
print("Available preprocessing variable sets:")
list_variables()

Available preprocessing variable sets:


,ID,Time,Brightness,Contrast,Threshold,Erosion,Dialation,Description
0,1,2026-01-16T10:32:29,-7.45,8.06,45,2,3,test


## Load Dataset and Initialize Paths

Select a dataset and establish input/output directory paths. Input images are expected in `{Death_type}/{Cell_type}_Green/` and `{Death_type}/{Cell_type}_Phase/` subdirectories. Background crops are saved to a new `Background/` directory.

In [24]:
"""Load dataset configuration and initialize paths."""
# Load dataset (using preset 1; modify as needed)
dataset = use_dataset(1)
print(f"Dataset: {dataset['Cell_type']} cells, {dataset['Death_type']} death type")
print(f"Image path: {dataset['Image_path']}")

# Construct input paths
GREEN_PATH = os.path.join(dataset['Image_path'], dataset['Death_type'], f"{dataset['Cell_type']}_Green/")
PHASE_PATH = os.path.join(dataset['Image_path'], dataset['Death_type'], f"{dataset['Cell_type']}_Phase/")

# Construct output path for background crops
IMAGES_BACKGROUND_DIR_PATH = os.path.join(dataset['Image_path'], "Background/")

# Create output directory
try:
    os.makedirs(IMAGES_BACKGROUND_DIR_PATH, exist_ok=True)
    print(f"✓ Background directory ready: {IMAGES_BACKGROUND_DIR_PATH}")
except OSError as e:
    print(f"✗ Error creating {IMAGES_BACKGROUND_DIR_PATH} directory: {e}")

Dataset: MEF cells, Necroptosis death type
Image path: Data
✓ Background directory ready: Data/Background/


In [31]:
"""Load preprocessing parameters (using preset 15; modify as needed)."""
variables = use_variables(1)
print("Loaded preprocessing parameters:")
variables

Loaded preprocessing parameters:


{'ID': 1,
 'Time': '2026-01-16T10:32:29',
 'Brightness': -7.45,
 'Contrast': 8.06,
 'Threshold': 45,
 'Erosion': 2,
 'Dialation': 3,
 'Description': 'test'}

## Cropping Parameters and Filtering Criteria

**Crop Dimensions:**
- Patch size: 128×128 pixels
- Horizontal overlap: 64 pixels (50%)
- Vertical overlap: 52 pixels (40.6%)
- Sliding strides: 64 px (horizontal), 76 px (vertical)

**Filtering Criteria:**
- Accept crops only if no cells detected (after processing)
- Accept crops only if image darkness < 40% mean intensity (dark background)
- Cap total collected: 10,000 patches per dataset

This ensures the background dataset contains diverse, cell-free regions representative of the imaging environment.

In [32]:
"""
Extract background image patches using sliding-window cropping (optimized).

Process:
1. Filter for image files (png, jpg, jpeg)
2. For each image: extract overlapping 128x128 patches
3. Apply preprocessing pipeline to each patch
4. Detect cells using bounding box analysis
5. Save patches with no detections and low mean intensity
"""

# Crop parameters (precompute constants)
CROP_WIDTH = 128
CROP_HEIGHT = 128
CROP_WIDTH_SHIFT = 64  # 128 - 64 = 64
CROP_HEIGHT_SHIFT = 76  # 128 - 52 = 76
INTENSITY_THRESHOLD = 40.0
MAX_PATCHES = 1000

# Collect and sort images
image_files = sorted([
    f for f in os.listdir(GREEN_PATH) 
    if f.lower().endswith(('png', 'jpg', 'jpeg'))
])
num_total_images = len(image_files)
num_patches_collected = 0

print(f"Processing {num_total_images} images...\n")

for img_idx, image_name in enumerate(image_files, 1):
    # Early exit if patch limit reached
    if num_patches_collected >= MAX_PATCHES:
        print(f"✓ Patch limit ({MAX_PATCHES}) reached. Stopping.")
        break
    
    print(f"{img_idx}/{num_total_images}: {image_name}", end='', flush=True)
    
    # Construct paths
    green_path = os.path.join(GREEN_PATH, image_name)
    phase_path = os.path.join(PHASE_PATH, image_name)
    
    # Read images
    img_processed, img_phase = read_image(green_path, phase_path)
    
    # Validate reads
    if img_processed is None or img_phase is None:
        print(" ⚠ (skip: read failed)\n")
        continue
    
    img_h, img_w = img_phase.shape[:2]
    
    # Generate sliding window coordinates (optimized: pre-compute all positions)
    x_positions = list(range(0, max(1, img_w - CROP_WIDTH + 1), CROP_WIDTH_SHIFT))
    if not x_positions or x_positions[-1] + CROP_WIDTH < img_w:
        x_positions.append(max(0, img_w - CROP_WIDTH))
    
    y_positions = list(range(0, max(1, img_h - CROP_HEIGHT + 1), CROP_HEIGHT_SHIFT))
    if not y_positions or y_positions[-1] + CROP_HEIGHT < img_h:
        y_positions.append(max(0, img_h - CROP_HEIGHT))
    
    num_patches_this_image = 0
    base_name = os.path.splitext(image_name)[0]
    
    # Extract and filter patches
    for wx, x in enumerate(x_positions):
        x_end = x + CROP_WIDTH
        for hy, y in enumerate(y_positions):
            y_end = y + CROP_HEIGHT
            
            # Vectorized slicing (no intermediate copies)
            patch_processed = img_processed[y:y_end, x:x_end]
            patch_phase = img_phase[y:y_end, x:x_end]
            
            # Apply preprocessing
            patch_dilated = process_image(
                patch_processed,
                variables['Contrast'],
                variables['Brightness'],
                variables['Threshold'],
                variables['Erosion'],
                variables['Dialation'],
                plot=False
            )
            
            # Detect cells
            bboxes = get_bboxes(patch_dilated)
            
            # Skip if cells detected (early exit saves computation)
            if len(bboxes) > 0:
                continue
            
            # Compute mean intensity
            mean_val = (np.mean(patch_dilated) / 255.0) * 100.0
            
            # Save only dark backgrounds
            if mean_val < INTENSITY_THRESHOLD:
                num_patches_collected += 1
                num_patches_this_image += 1
                
                output_name = f"{base_name}_{wx}_{hy}.png"
                output_path = os.path.join(IMAGES_BACKGROUND_DIR_PATH, output_name)
                cv2.imwrite(output_path, patch_phase)
                
                # Early exit if limit reached
                if num_patches_collected >= MAX_PATCHES:
                    break
        
        if num_patches_collected >= MAX_PATCHES:
            break
    
    print(f" ({num_patches_this_image} patches saved)\n")

print(f"\n✓ Complete. Collected {num_patches_collected} background patches.")
print(f"  Output directory: {IMAGES_BACKGROUND_DIR_PATH}")

Processing 13 images...

1/13: VID856_B4_1_00d00h00m.png (211 patches saved)

2/13: VID856_B4_1_00d02h00m.png (116 patches saved)

3/13: VID856_B4_1_00d04h00m.png (29 patches saved)

4/13: VID856_B4_1_00d06h00m.png (1 patches saved)

5/13: VID856_B4_1_00d08h00m.png (0 patches saved)

6/13: VID856_B4_1_00d10h00m.png (1 patches saved)

7/13: VID856_B4_1_00d12h00m.png (2 patches saved)

8/13: VID856_B4_1_00d14h00m.png (5 patches saved)

9/13: VID856_B4_1_00d16h00m.png (10 patches saved)

10/13: VID856_B4_1_00d18h00m.png (19 patches saved)

11/13: VID856_B4_1_00d20h00m.png (27 patches saved)

12/13: VID856_B4_1_00d22h00m.png (44 patches saved)

13/13: VID856_B4_1_01d00h00m.png (42 patches saved)


✓ Complete. Collected 507 background patches.
  Output directory: Data/Background/
